[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Visualización de datos con *Pandas*.

## Contexto de los datos.

Para este capítulo se empleará una copia fija del indicador del Banco Mundial que se descargó en el capítulo 26: el producto interno bruto (PIB) anual por país, en dólares estadounidenses a precios constantes de 2015 (indicador `NY.GDP.MKTP.KD`).

Esta decisión permite enfocarse en las técnicas de visualización con *Pandas* sin depender de fuentes externas que pueden cambiar con el tiempo (enlaces, estructura de los datos o revisiones de las cifras).

**Fuente:** *World Bank Open Data*, Banco Mundial, licencia CC BY 4.0 (https://data.worldbank.org/indicator/NY.GDP.MKTP.KD), consultado el 27 de noviembre de 2026. El archivo se genera con `python scripts/descargar_banco_mundial.py`.

* La siguiente celda importará *Pandas*.

In [ ]:
import pandas as pd

## El atributo ```plot``` de los objetos `DataFrame` y `Series`.

Este atributo contiene una colección de métodos que permiten desplegar gráficos descriptivos básicos basados en *Matplotlib*.

Algunos de los métodos más comunes son:

* ```pd.plot.line()```: para gráficos de líneas.
* ```pd.plot.bar()```: para gráficos de barras.
* ```pd.plot.barh()```: para gráficos de barras horizontales.
* ```pd.plot.pie()```: para gráficos de pastel.
* ```pd.plot.hist()```: para histogramas.
* ```pd.plot.scatter()```: para gráficos de dispersión.
* ```pd.plot.box()```: para diagramas de caja.
* ```pd.plot.area()```: para gráficos de área.

En este capítulo se introducen los métodos básicos. Los capítulos siguientes profundizan en cada tipo de gráfica.

### Creación de gráficos con `plot()`.

Para crear un gráfico con el método ```plot()```, se puede seguir la siguiente sintaxis:

```
<obj>.plot(kind='tipo_de_grafico', x='columna_x', y='columna_y', title='Título del gráfico', xlabel='Etiqueta eje x', ylabel='Etiqueta eje y', **kwargs)
```

Donde:
* ```<obj>```: puede ser un objeto `DataFrame` o `Series` de *Pandas*.
* ```kind```: especifica el tipo de gráfico que se desea crear (por ejemplo, 'line', 'bar', 'hist', 'scatter', 'box').
* ```x```: indica la columna del DataFrame que se utilizará para el eje x.
* ```y```: indica la columna del DataFrame que se utilizará para el eje y.
* ```title```: establece el título del gráfico.
* ```xlabel```: establece la etiqueta del eje x.
* ```ylabel```: establece la etiqueta del eje y.

Además de estos parámetros básicos, el método `plot()` acepta una amplia variedad de argumentos adicionales (`**kwargs`) derivados de *Matplotlib* que permiten personalizar aún más el gráfico, como colores, estilos de línea, tamaños de marcador, leyendas, entre otros.

Los próximos capítulos explorarán varias características avanzadas de visualización de *Matplotlib*.

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html

## Ejemplo de uso.

* El archivo `data/pib_banco_mundial.csv` contiene el PIB anual de 23 países de las siete regiones del Banco Mundial y el agregado mundial, de 1990 a 2025, en formato largo: un renglón por país y año.

| Columna | Contenido |
| :--- | :--- |
| `codigo` | Código del país de tres letras (norma ISO 3166-1 alfa-3). El agregado mundial tiene el código `WLD`. |
| `pais` | Nombre del país. |
| `region` | Región del Banco Mundial. El agregado mundial tiene la región `Agregados`. |
| `año` | Año. |
| `pib` | PIB en dólares estadounidenses a precios constantes de 2015. |

* La siguiente celda creará el *dataframe* `datos` a partir del archivo `data/pib_banco_mundial.csv`.

In [ ]:
import os
datos = pd.read_csv(os.path.join('data', 'pib_banco_mundial.csv'), encoding='utf-8')

* La siguiente celda mostrará el *dataframe* `datos`.

In [ ]:
datos

* La siguiente celda mostrará el tipo de cada columna. `pd.read_csv()` infiere que `año` contiene enteros y `pib` números de punto flotante.

In [ ]:
datos.info()

### 1. Adecuación de los datos.

* La siguiente celda agregará la columna `pib_mmd` con el PIB en miles de millones de dólares, para que las gráficas muestren cifras más fáciles de leer.

In [ ]:
datos['pib_mmd'] = datos['pib'] / 1e9
datos.head()

### 2. Reestructuración a formato ancho.

Para graficar países y años conviene un *dataframe* con un renglón por país y una columna por año. El método `df.pivot()`, estudiado en el capítulo 21, transforma el formato largo en formato ancho.

* La siguiente celda creará el *dataframe* `pib` con el nombre del país como índice y los años como columnas.

In [ ]:
pib = datos.pivot(index='pais', columns='año', values='pib_mmd')
pib

### 3. Separación del agregado mundial.

El renglón `Mundo` es un agregado: suma la producción de todos los países del mundo, no sólo la de los 23 países del archivo. Para comparar países entre sí se separa del resto.

* La siguiente celda guardará en `paises` todos los renglones de `pib` excepto `Mundo`, usando `df.drop()`.

In [ ]:
paises = pib.drop(index='Mundo')
paises.shape

### 4. Algunas gráficas básicas.

A continuación se muestran algunos ejemplos de gráficas básicas que se pueden crear con el método `plot()` de *Pandas*.

* La siguiente celda mostrará un gráfico de barras con el PIB de cada país en 2024, ordenado de mayor a menor. El acceso `paises[2024]` obtiene directamente una *Series* con los países como índice.

In [ ]:
paises[2024].sort_values(ascending=False).plot.bar(
    title='PIB por país - 2024',
    xlabel='País',
    ylabel='Miles de millones de dólares de 2015',
    figsize=(14, 5),
    rot=45
)

* La siguiente celda mostrará un gráfico de barras horizontales con el mismo corte de datos utilizando el método `plot.barh()`.

In [ ]:
paises[2024].sort_values().plot.barh(
    title='PIB por país - 2024',
    xlabel='Miles de millones de dólares de 2015',
    figsize=(8, 9)
)

* La siguiente celda agrupará los países por región con `df.groupby()`, estudiado en el capítulo 24, sumará el PIB de 2024 de cada región y mostrará el resultado en un gráfico de barras horizontales. El agregado mundial se excluye con la condición `datos['codigo'] != 'WLD'`.

In [ ]:
por_region = (
    datos[(datos['año'] == 2024) & (datos['codigo'] != 'WLD')]
    .groupby('region')['pib_mmd']
    .sum()
    .sort_values()
)
por_region.plot.barh(
    title='PIB de los países del archivo por región - 2024',
    xlabel='Miles de millones de dólares de 2015',
    figsize=(8, 4)
)

### 5. Creación de un nuevo *dataframe*.

Para graficar la evolución en el tiempo, el método `plot()` usa el índice como eje horizontal y dibuja una línea por columna. Por ello conviene un *dataframe* con los años en el índice y los países en las columnas.

* La siguiente celda creará el *dataframe* `datos_anuales` como la transpuesta de `pib`, de modo que el año quede en el índice y cada país en una columna.

In [ ]:
datos_anuales = pib.transpose()
datos_anuales

### 6. Visualización con `datos_anuales`.

A continuación se muestran algunos ejemplos de visualización de los datos del *dataframe* `datos_anuales` por medio de los métodos del atributo `plot`.

Los métodos de visualización de datos del atributo `plot` permiten personalizar los gráficos con diferentes opciones, como el título, las etiquetas de los ejes, el tamaño de la figura, entre otras, aprovechando los parámetros de los métodos de *Matplotlib*.

En este caso, se utilizarán los parámetros por defecto para mostrar la simplicidad de uso de los métodos del atributo `plot`.

Los parámetros de *Matplotlib* se estudiarán en capítulos posteriores, pero la documentación oficial está disponible en:

https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html

* La siguiente celda mostrará un gráfico de líneas con la evolución anual del PIB mundial (columna `'Mundo'`) utilizando el método `plot()`.

In [ ]:
datos_anuales['Mundo'].plot(
    kind="line",
    title='PIB mundial',
    xlabel='Año',
    ylabel='Miles de millones de dólares de 2015',
    figsize=(14, 5),
    xticks=datos_anuales.index,
    rot=45,
    grid=True
)

* La siguiente celda mostrará en un mismo gráfico de líneas la evolución del PIB de cuatro países de distintas regiones. Al seleccionar varias columnas, `plot()` dibuja una línea por columna y agrega una leyenda.

In [ ]:
datos_anuales[['Brasil', 'India', 'Japón', 'Nigeria']].plot(
    kind="line",
    title='PIB de cuatro países',
    xlabel='Año',
    ylabel='Miles de millones de dólares de 2015',
    figsize=(14, 5),
    grid=True
)

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>